In [1]:
from utils import *
from ops import *
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialisation
seq = RenderSequence("DATA/Musicroom/", device=device)

# Accès direct à la couleur de l'image à l'index 2 (0002)
color_frame_2 = seq.get_frame_pass(2, "color")
print(f"Shape color: {color_frame_2.shape}") 

# Récupérer toutes les passes d'une frame d'un coup
frame_0 = seq[0] 
print(frame_0.keys()) 

Séquence chargée : 3 frames trouvées.
Shape color: torch.Size([1, 3, 1080, 1920])
dict_keys(['ref', 'color', 'albedo', 'normal', 'opacity', 'color2', 'emissive', 'envLight', 'linearZ', 'mvec', 'pnFwidth', 'ref_emissive', 'ref_envLight'])


## CROSS-REGRESSION

In [3]:
yA = seq.get_frame_pass(0, "color")
yB = seq.get_frame_pass(0, "color2")
normals = seq.get_frame_pass(0, "normal")
textures = seq.get_frame_pass(0, "albedo")

print(normals.shape)
print(textures.shape)

alphaA, betaA, alphaB, betaB, sigmaA, sigmaB = compute_alpha_beta(yA, yB, textures, normals, stride=4, window_size=17)
f_tildeA = compute_f_tilde(alphaA, betaA, yB, textures, normals, sigmaA)
f_tildeB = compute_f_tilde(alphaB, betaB, yA, textures, normals, sigmaB)
save_tensor_as_exr(f_tildeA, "./results/f_tildeA_0000.exr")
save_tensor_as_exr(f_tildeB, "./results/f_tildeB_0000.exr")

torch.Size([1, 3, 1080, 1920])
torch.Size([1, 3, 1080, 1920])
Fichier sauvegardé : ./results/f_tildeA_0000.exr
Fichier sauvegardé : ./results/f_tildeB_0000.exr


In [4]:
input = preprocessing(f_tildeA, f_tildeB, textures, normals)
print("Input to network:", input.shape)
model = SmallUNet(in_channels=15, out_channels=6).to(device)
theta = model(input)
#sanity_check()

Input to network: torch.Size([1, 15, 1080, 1920])
Shape after enc1: torch.Size([1, 8, 1080, 1920])
Shape after pool1: torch.Size([1, 8, 540, 960])
Shape after enc2: torch.Size([1, 16, 540, 960])
Shape after pool2: torch.Size([1, 16, 270, 480])
Shape after enc3 (bottleneck): torch.Size([1, 32, 270, 480])
Shape after up2: torch.Size([1, 32, 540, 960])
Shape after concat with s2: torch.Size([1, 48, 540, 960])
Shape after dec2: torch.Size([1, 16, 540, 960])
Shape after up1: torch.Size([1, 16, 1080, 1920])
Shape after cat: torch.Size([1, 24, 1080, 1920])
Shape after dec1: torch.Size([1, 8, 1080, 1920])


In [ ]:
positions = create_position_buffer(f_tildeA.shape[0], f_tildeA.shape[2], f_tildeA.shape[3], device=device)

print(positions[0][0,0,:])

torch.cuda.empty_cache()

with torch.no_grad():
    _, _, final_image = final_spatiotemporal_pipeline(
        theta, f_tildeA, f_tildeB, 
        textures, normals, positions, 
        f_prev=None, 
        kernel_size=11, 
        tile_size=256
    )

save_tensor_as_exr(final_image, "./results/final_image_0000.exr")

tensor([0.0000e+00, 5.2110e-04, 1.0422e-03,  ..., 9.9896e-01, 9.9948e-01,
        1.0000e+00], device='cuda:0')
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: torch.Size([1, 3, 266, 266])
Compute weights for f_in shape: to

# Whole pipeliine

In [2]:
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seq = RenderSequence("DATA/Musicroom/", device=device)

color_frame_0 = seq.get_frame_pass(0, "color")

Séquence chargée : 101 frames trouvées.


In [3]:
W, H = color_frame_0.shape[3], color_frame_0.shape[2]
# Créer un buffer de positions
G_positions = create_position_buffer(color_frame_0.shape[0], H, W, device=device)

### Train loop

In [4]:
def train():
    model = SmallUNet(in_channels=15, out_channels=6).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    optimizer.zero_grad()

    # Frame précédent
    f_tilde_prev_A = None
    f_tilde_prev_B = None
    f_denoise_prev = None

    for i in range(len(seq)):
        yA = seq.get_frame_pass(i, "color")
        yB = seq.get_frame_pass(i, "color2")
        G_normals = seq.get_frame_pass(i, "normal")
        G_textures = seq.get_frame_pass(i, "albedo")
        mvec = seq.get_frame_pass(i, "mvec")
        
        # Cross-Regression
        alphaA, betaA, alphaB, betaB, sigmaA, sigmaB = compute_alpha_beta(yA, yB, G_textures, G_normals, stride=4, window_size=17)
        f_tildeA = compute_f_tilde(alphaA, betaA, yB, G_textures, G_normals, sigmaA)
        f_tildeB = compute_f_tilde(alphaB, betaB, yA, G_textures, G_normals, sigmaB)
        save_tensor_as_exr(f_tildeA, f"./results/f_tildeA_{i:04d}.exr")
        save_tensor_as_exr(f_tildeB, f"./results/f_tildeB_{i:04d}.exr")


        # Prétraitement et passage dans le réseau
        input = preprocessing(f_tildeA, f_tildeB, G_textures, G_normals, f_prev_denoised=f_denoise_prev)
        full_theta = model(input)
        
        # Passage dans le filtre spatio-temporel
        f_hat_A, f_hat_B, final_image = final_spatiotemporal_pipeline(
            full_theta, f_tildeA, f_tildeB, 
            G_textures, G_normals, G_positions, 
            f_prev=f_denoise_prev, 
            kernel_size=11, 
            tile_size=1024
        )

        # Spatial Loss
        spatial_loss = spatial_loss_fn(f_hat_A, f_hat_B, f_tildeA, f_tildeB)

        # Temporal Loss
        mvec_norm = prepare_motion_vectors(mvec, H, W) # Normaliser mvec
        temporal_loss = temporal_loss_fn(f_hat_A, f_hat_B, f_tilde_prev_A, f_tilde_prev_B, mvec_norm)
        
        # Total Loss
        total_loss = spatial_loss + temporal_loss
        total_loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        f_tilde_prev_A = f_tildeA.detach()
        f_tilde_prev_B = f_tildeB.detach()
        f_denoise_prev = final_image.detach() # Stocker l'image finale débruitée pour la prochaine itération

        del f_tildeA, f_tildeB, full_theta, final_image, f_hat_A, f_hat_B, total_loss
        
        # Sauvegarde de l'image finale pour inspection
        save_tensor_as_exr(f_denoise_prev, f"./results/final_image_{i:04d}.exr")

train()

Fichier sauvegardé : ./results/f_tildeA_0000.exr
Fichier sauvegardé : ./results/f_tildeB_0000.exr
Fichier sauvegardé : ./results/final_image_0000.exr
Fichier sauvegardé : ./results/f_tildeA_0001.exr
Fichier sauvegardé : ./results/f_tildeB_0001.exr
Fichier sauvegardé : ./results/final_image_0001.exr
Fichier sauvegardé : ./results/f_tildeA_0002.exr
Fichier sauvegardé : ./results/f_tildeB_0002.exr
Fichier sauvegardé : ./results/final_image_0002.exr
Fichier sauvegardé : ./results/f_tildeA_0003.exr
Fichier sauvegardé : ./results/f_tildeB_0003.exr
Fichier sauvegardé : ./results/final_image_0003.exr
Fichier sauvegardé : ./results/f_tildeA_0004.exr
Fichier sauvegardé : ./results/f_tildeB_0004.exr
Fichier sauvegardé : ./results/final_image_0004.exr
Fichier sauvegardé : ./results/f_tildeA_0005.exr
Fichier sauvegardé : ./results/f_tildeB_0005.exr
Fichier sauvegardé : ./results/final_image_0005.exr
Fichier sauvegardé : ./results/f_tildeA_0006.exr
Fichier sauvegardé : ./results/f_tildeB_0006.exr
Fi